<a href="https://colab.research.google.com/github/OuhmadMohamed/DI_Bootcamp/blob/main/Week9/Day1/daily_chalange_w9_d1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Understanding BERT and XLM-RoBERTa
Objective

Learn how transformer models work and how they process text with tokenization.

Key Concepts

BERT (Bidirectional Encoder Representations from Transformers)

Reads text bidirectionally (left → right and right → left).

Uses WordPiece tokenization.

Pre-trained on large English corpora.

Common models:

bert-base-uncased

bert-base-cased

XLM-RoBERTa

Multilingual version of RoBERTa.

Trained on 100+ languages.

Uses SentencePiece tokenization.

Strong for multilingual classification.

Import tokenizers

In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer


Task 2: Tokenizing Text
Objective

Convert sentences into tokenized inputs and explore token types.

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

text = "Transformers are amazing for NLP tasks."

encoded = bert_tokenizer.encode_plus(text)
encoded


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

{'input_ids': [101, 19081, 2024, 6429, 2005, 17953, 2361, 8518, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
bert_tokenizer.decode(encoded["input_ids"])


'[CLS] transformers are amazing for nlp tasks. [SEP]'

In [ ]:
encoded_pair = bert_tokenizer.encode_plus(
    "The sky is blue",
    "The ocean is blue"
)
encoded_pair

{'input_ids': [101, 1996, 3712, 2003, 2630, 102, 1996, 4153, 2003, 2630, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
bert_tokenizer.decode(encoded_pair["input_ids"])


'[CLS] the sky is blue [SEP] the ocean is blue [SEP]'

Task 3: Preparing Input Data for the Model
Objective

Pad, truncate, and understand special tokens.

In [ ]:
encoded = bert_tokenizer.encode_plus(
    text,
    max_length=32,
    padding="max_length",
    return_attention_mask=True
)
encoded

{'input_ids': [101, 19081, 2024, 6429, 2005, 17953, 2361, 8518, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [ ]:
bert_tokenizer.decode(encoded["input_ids"])

'[CLS] transformers are amazing for nlp tasks. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]'

In [ ]:
bert_tokenizer.special_tokens_map


{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [ ]:
bert_tokenizer.vocab_size


30522

✅ Task 4: Loading and Exploring the Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
df.head()


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [ ]:
df.shape


(12120, 6)

✅ Task 5: Creating Cross-Validation Folds

In [ ]:
from sklearn.model_selection import StratifiedKFold

X = df[["premise", "hypothesis"]]
y = df["label"]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_folds = []
val_folds = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    train_data = df.iloc[train_idx]
    val_data = df.iloc[val_idx]

    train_folds.append(train_data)
    val_folds.append(val_data)

    print(f"Fold {fold+1}:")
    print("Train size:", train_data.shape)
    print("Validation size:", val_data.shape)


Fold 1:
Train size: (9696, 6)
Validation size: (2424, 6)
Fold 2:
Train size: (9696, 6)
Validation size: (2424, 6)
Fold 3:
Train size: (9696, 6)
Validation size: (2424, 6)
Fold 4:
Train size: (9696, 6)
Validation size: (2424, 6)
Fold 5:
Train size: (9696, 6)
Validation size: (2424, 6)
